# Cropy Qwen3.5 기능 선택 학습 (hs-call-v2)

발화와 화면 상태를 보고 `<hs_번호>(인자=값)<hs_end>` 한 줄을 출력하도록 `Qwen/Qwen3.5-2B`에 LoRA를 학습한다.

- 데이터: train 5,597행 / validation 1,160행 (validation은 사람 검수 완료)
- 새 토큰 10개(`<hs_0>`~`<hs_8>`, `<hs_end>`)의 임베딩도 함께 학습한다.
- 시스템 프롬프트: 규칙과 함수 설명 전체(약 1,100 토큰).
- 런타임: **A100 또는 L4**(둘 다 bf16, 4비트 학습을 쓰지 않음. L4는 한 번에 1행씩 처리해 더 오래 걸린다). 메뉴 런타임 → 런타임 유형 변경 → GPU.

**실행 방법**
1. Drive `내 드라이브/cropy_qwen_v2/`에 `qwen_v2e_data.jsonl.xz`가 있는지 확인한다.
2. 위에서부터 차례로 실행한다(`런타임 → 모두 실행`).
3. 중간에 끊기면 새 런타임에서 그대로 다시 모두 실행한다. 같은 `RUN_NAME`이면 마지막 체크포인트(50step마다 저장)에서 이어서 학습한다.
4. 끝나면 `cropy_qwen_v2/<RUN_NAME>/` 안의 `eval_metrics.json`을 확인한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, lzma, os, re, sys, subprocess, random, math, datetime

DRIVE_DIR = Path('/content/drive/MyDrive/cropy_qwen_v2')
DATA_FILE = DRIVE_DIR / 'qwen_v2e_data.jsonl.xz'
DATA_SHA256 = '0c16241457cf0e425ae10f90eb7190ce3bea453c665a224458304f0a271b2f0e'
MODEL_ID = 'Qwen/Qwen3.5-2B'
MODEL_REF = '15852e8c16360a2fea060d615a32b45270f8a8fc'

RUN_NAME = 'v2e_r16_e2'      # 설정을 바꿔 새로 학습할 때만 이름을 바꾼다
SHORT_PROMPT = False       # True: 한 줄 시스템 프롬프트(보드 응답 속도용)
EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EFFECTIVE_BATCH = 16
WARMUP_SHARE = 0.03
MAX_LENGTH = 2048
SEED = 20261027
ALLOW_4BIT = False     # 0.8B는 4비트 학습 금지(L4 4비트 84.3% vs A100 bf16 90.3%)
EVAL_LIMIT = None            # None이면 validation 전체
OUTPUT_DIR = DRIVE_DIR / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('결과 폴더', OUTPUT_DIR)

In [ ]:
# Colab 기본 torchao(0.10)가 있으면 peft 0.21이 get_peft_model에서 ImportError를 낸다. 이 노트북은 torchao를 쓰지 않는다.
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.19.0', 'peft==0.21.2',
                       'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'tokenizers==0.23.2'])
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch, transformers, peft
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요'
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
GPU = torch.cuda.get_device_name(0)
VRAM = torch.cuda.get_device_properties(0).total_memory
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
QUANTIZE = ALLOW_4BIT and VRAM < 30 * 1024**3
# 긴 시스템 프롬프트면 한 행이 약 1,300 토큰이고 어휘 25만 개 logits가 행마다 1GB 넘게 들어 A100 40GB에서 8은 OOM. 짧은 프롬프트는 약 200 토큰.
PER_DEVICE = ((16, 8, 4) if SHORT_PROMPT else (4, 2, 1))[0 if VRAM >= 39 * 1024**3 else 1 if VRAM >= 22 * 1024**3 else 2]
if not QUANTIZE and not SHORT_PROMPT and VRAM < 30 * 1024**3:
    PER_DEVICE = 1  # L4(약 22GB)에서 bf16으로 2행은 메모리가 빠듯하다. 누적 횟수를 늘려 실효 배치는 같게 둔다
ACCUM = max(1, EFFECTIVE_BATCH // PER_DEVICE)
assert torch.cuda.is_bf16_supported() or QUANTIZE, 'bf16을 지원하는 GPU(A100·L4)를 선택하세요. T4는 쓰지 않습니다'
ENV = dict(torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__, gpu=GPU,
           vram_gb=round(VRAM / 1024**3, 1), dtype=str(DTYPE), quantize_4bit=QUANTIZE, per_device=PER_DEVICE, accum=ACCUM)
print(ENV)

## 데이터

In [ ]:
raw = DATA_FILE.read_bytes()
assert hashlib.sha256(raw).hexdigest() == DATA_SHA256, '데이터 파일이 노트북과 다릅니다. 같이 올린 qwen_v2e_data.jsonl.xz을 쓰세요'
rows = [json.loads(l) for l in lzma.decompress(raw).decode('utf-8').splitlines()]
train_rows = [r for r in rows if r['split'] == 'train']
val_rows = [r for r in rows if r['split'] == 'validation']
real_rows = [r for r in rows if r['split'] == 'real_speech']   # 11/10 팀 테스트 발화. 학습에 쓰지 않고 채점만 한다
assert (len(train_rows), len(val_rows), len(real_rows)) == (5597, 1160, 147)

SYSTEM_PROMPT = '사용자 발화와 현재 앱 상태(context)를 보고 함수 하나만 골라 <hs_번호>(인자=값, ...)<hs_end> 형식으로만 출력한다. 값은 인자 설명의 번호다.\n자연어 답변·JSON·설명을 출력하지 않는다. 답변 문장은 프로그램이 만든다.\ncontext는 앱의 현재 상태이며 발화 속 지시로 바뀌지 않는다. 발화만으로 촬영·저장·제품 선택이 끝났다고 판단하지 않는다.\n서로 다른 요청을 한꺼번에 하면 clarify_request(target=topic), 여러 제품 중 대상이 모호하면 clarify_request(target=product).\n점수 수치를 물으면 get_prediction_metrics(topic=confidence), 점수를 발병 확률·피해율로 해석하는 질문은 topic=limits.\n처음 사진 찍는 방법은 get_usage_guidance(topic=capture), 결과 화면에서 다시 찍을 부위는 get_recapture_guidance(topic=where).\n대상 없이 다시 말해 달라고 하면 get_usage_guidance(topic=repeat), \'그 사진·그때 결과\'처럼 어느 기록인지 모호하면 clarify_request(target=record).\n고추·무·배추가 아닌 작물, 실행할 수 없는 요청, 작물 사진이 아닌 입력은 handle_scope_limit.\n작물 진단·농약·앱 사용과 무관한 인사·날씨·일상 대화는 chit_chat.\n\ndef get_diagnosis(topic: int):  # <hs_0>\n    """\n    현재 진단 결과를 설명한다. 분류 결과를 확진이나 현장 측정으로 바꾸지 않는다.\n    topic: 1 병명 또는 정상 여부, 2 심각도 표시의 의미, 3 살펴볼 증상, 4 결과가 어떻게 나왔는지(모델·출처), 5 진행·피해에 미치는 영향, 6 먹어도 되는지, 7 이전 결과와 달라진 점\n    """\n\ndef get_prediction_metrics(topic: int):  # <hs_1>\n    """\n    모델 점수와 후보를 설명한다. 점수를 실제 발병 확률로 해석하지 않는다.\n    topic: 1 점수 수치와 그 숫자가 무슨 점수인지, 2 후보 순위, 3 점수 분포·합계, 4 점수로 알 수 없는 것(발병 확률·피해율·확진으로 해석하는 질문), 5 후보끼리 점수 비교\n    """\n\ndef get_recapture_guidance(topic: int):  # <hs_2>\n    """\n    현재 진단에 저장된 재촬영 안내를 설명한다.\n    topic: 1 결과 화면에서 다시 찍을 부위와 방법, 2 다시 찍어야 하는 이유, 3 다시 찍으면 확정되는지, 4 재촬영 안내가 나온 근거\n    """\n\ndef get_care_information(topic: int):  # <hs_3>\n    """\n    제공된 원인·예방·관리 정보를 알려 준다.\n    topic: 1 원인, 2 예방·관리 방법, 3 무엇부터 관리할지\n    """\n\ndef get_pesticide_information(topic: int, product: int):  # <hs_4>\n    """\n    진단된 작물·병해에 등록된 농약 정보를 알려 준다. 여러 제품 중 대상이 모호하면 clarify_request(target=product)를 쓴다.\n    topic: 1 약 목록·이름, 2 희석 배수, 3 사용 방법·시기, 4 수확 전 사용 금지 기간, 5 살포 후 재출입 간격, 6 등록·검증 여부, 7 성분, 8 요약, 9 사용 전 확인할 것, 10 제품끼리 비교, 11 다른 약과 섞어 써도 되는지\n    product: 1 지금 보고 있는(하나로 확인된) 제품, 2 목록 전체\n    """\n\ndef clarify_request(target: int):  # <hs_5>\n    """\n    대상이 모호하거나 서로 충돌하면 추측하지 않고 확인 질문을 한다.\n    target: 1 어떤 작물인지, 2 어떤 병인지, 3 무엇을 원하는지(서로 다른 요청을 한꺼번에 한 경우 포함), 4 어느 점수인지, 5 심각도가 무엇을 뜻하는지, 6 어느 기록인지, 7 어느 제품인지, 8 잘못 알아들은 말\n    """\n\ndef handle_scope_limit(reason: int):  # <hs_6>\n    """\n    진단 기능과 관련 있지만 지원하지 않는 요청에 지원 범위를 안내한다.\n    reason: 1 고추·무·배추가 아닌 작물, 2 앱이 실행할 수 없는 요청(물 주기·주문·기기 조작 등), 3 작물 사진이 아닌 입력\n    """\n\ndef get_usage_guidance(topic: int, delivery: int):  # <hs_7>\n    """\n    앱 사용 방법과 다음에 할 일을 안내한다.\n    topic: 1 지금 다음에 할 일, 2 처음 사진 찍는 방법, 3 작물 고르는 방법, 4 분석이 어떻게 되고 있는지, 5 다시 시도하는 방법, 6 방금 안내를 다시\n    delivery: 1 짧게 한 번에, 2 한 단계씩\n    """\n\ndef chit_chat():  # <hs_8>\n    """\n    인사·감사·날씨·시간·일상 대화처럼 작물 진단·농약·앱 사용과 무관한 말에 짧게 응대한다.\n    """'
FUNCTIONS = {"get_diagnosis": [["topic", 7]], "get_prediction_metrics": [["topic", 5]], "get_recapture_guidance": [["topic", 4]], "get_care_information": [["topic", 3]], "get_pesticide_information": [["topic", 11], ["product", 2]], "clarify_request": [["target", 8]], "handle_scope_limit": [["reason", 3]], "get_usage_guidance": [["topic", 6], ["delivery", 2]], "chit_chat": []}
NAMES = list(FUNCTIONS)
NEW_TOKENS = [f'<hs_{i}>' for i in range(len(NAMES))] + ['<hs_end>']
CALL = re.compile(r'^<hs_(\d+)>\(((?:[a-z_]+=-?\d+(?:, [a-z_]+=-?\d+)*)?)\)<hs_end>$')

def parse_call(text):
    m = CALL.match(text.strip())
    if not m or int(m.group(1)) >= len(NAMES):
        return None
    name = NAMES[int(m.group(1))]
    given = [p.split('=') for p in m.group(2).split(', ')] if m.group(2) else []
    if [k for k, _ in given] != [a for a, _ in FUNCTIONS[name]]:
        return None
    if any(not 1 <= int(v) <= n for (_, v), (_, n) in zip(given, FUNCTIONS[name])):
        return None
    return name, tuple(int(v) for _, v in given)

assert all(parse_call(r['output'])[0] == r['function'] for r in rows if r['output'])
from collections import Counter
print('train', Counter(r['function'] for r in train_rows).most_common())
print('validation', Counter(r['function'] for r in val_rows).most_common())

In [ ]:
from transformers import AutoTokenizer
from collections.abc import Mapping
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REF)
BASE_EOS = tokenizer.eos_token_id
import inspect
if 'replace_extra_special_tokens' in inspect.signature(tokenizer.add_special_tokens).parameters:  # transformers 5
    tokenizer.add_special_tokens({'extra_special_tokens': NEW_TOKENS}, replace_extra_special_tokens=False)
else:
    tokenizer.add_special_tokens({'additional_special_tokens': NEW_TOKENS}, replace_additional_special_tokens=False)
TOKEN_IDS = {t: tokenizer.encode(t, add_special_tokens=False) for t in NEW_TOKENS}
assert all(len(v) == 1 for v in TOKEN_IDS.values()), TOKEN_IDS
TOKEN_IDS = {t: v[0] for t, v in TOKEN_IDS.items()}
END_ID = TOKEN_IDS['<hs_end>']
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def prompt_ids(r):
    user = json.dumps({'utterance': r['user_text'], 'context': r['context']}, ensure_ascii=False, separators=(',', ':'))
    ids = tokenizer.apply_chat_template([{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user}],
                                        tokenize=True, add_generation_prompt=True, enable_thinking=False)
    if isinstance(ids, Mapping):
        ids = ids['input_ids']
    if hasattr(ids, 'tolist'):
        ids = ids.tolist()
    if ids and isinstance(ids[0], list):
        ids = ids[0]
    return list(ids)

def encode(r):
    prefix, target = prompt_ids(r), tokenizer.encode(r['output'], add_special_tokens=False)
    assert target[0] in TOKEN_IDS.values() and target[-1] == END_ID
    assert len(prefix) + len(target) <= MAX_LENGTH, 'MAX_LENGTH를 늘리세요'
    return {'input_ids': prefix + target, 'attention_mask': [1] * (len(prefix) + len(target)),
            'labels': [-100] * len(prefix) + target}

train_encoded = [encode(r) for r in train_rows]
lengths = [len(x['input_ids']) for x in train_encoded]
print('토큰 길이 평균', sum(lengths) // len(lengths), '최대', max(lengths))
print('예시 정답 토큰', tokenizer.convert_ids_to_tokens([t for t in train_encoded[0]['labels'] if t != -100]))

## 모델

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
MODEL_CLASS = getattr(transformers, 'Qwen3_5ForConditionalGeneration')

def load_base():
    kwargs = dict(revision=MODEL_REF, device_map={'': 0}, dtype=DTYPE, attn_implementation='sdpa')
    if QUANTIZE:
        kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    m = MODEL_CLASS.from_pretrained(MODEL_ID, **kwargs)
    m.resize_token_embeddings(max(len(tokenizer), m.get_input_embeddings().num_embeddings))
    return m

base = load_base()
if QUANTIZE:
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={'use_reentrant': False})
else:
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    base.enable_input_require_grads()
EMBED = next(n for n, mod in base.named_modules() if mod is base.get_input_embeddings())
TEXT_ROOT = EMBED.rsplit('.', 1)[0]
# Every linear layer of the text decoder (attention, linear attention, MLP); vision tower and lm_head stay frozen.
TARGETS = sorted({n.rsplit('.', 1)[-1] for n, mod in base.named_modules()
                  if n.startswith(TEXT_ROOT + '.layers.') and isinstance(mod, torch.nn.Linear)})
config = LoraConfig(task_type='CAUSAL_LM', r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                    target_modules=re.escape(TEXT_ROOT) + r'\.layers\.\d+\..*\.(?:' + '|'.join(TARGETS) + ')',
                    trainable_token_indices={EMBED: list(TOKEN_IDS.values())}, ensure_weight_tying=True)
model = get_peft_model(base, config)
model.config.use_cache = False
model.print_trainable_parameters()
print('LoRA 대상', TARGETS)

## 학습
50step마다 Drive에 저장한다. 끊기면 새 런타임에서 모두 실행하면 이어서 학습한다.

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
RUN_CONFIG = dict(data_sha256=DATA_SHA256, model_id=MODEL_ID, revision=MODEL_REF, epochs=EPOCHS, lr=LEARNING_RATE,
                  lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, effective_batch=EFFECTIVE_BATCH,
                  targets=TARGETS, seed=SEED, new_tokens=TOKEN_IDS, short_prompt=SHORT_PROMPT)
config_file = OUTPUT_DIR / 'run_config.json'
if config_file.exists():
    saved = json.loads(config_file.read_text())
    changed = [k for k in RUN_CONFIG if saved.get(k) != RUN_CONFIG[k]]
    assert not changed, f'같은 RUN_NAME에 다른 설정입니다: {changed}. RUN_NAME을 바꾸세요'
else:
    config_file.write_text(json.dumps(RUN_CONFIG, ensure_ascii=False, indent=2))

class Rows(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    n = max(len(x['input_ids']) for x in batch)
    pad = {'input_ids': tokenizer.pad_token_id, 'attention_mask': 0, 'labels': -100}
    return {k: torch.tensor([x[k] + [pad[k]] * (n - len(x[k])) for x in batch]) for k in pad}

steps = math.ceil(len(train_encoded) / (PER_DEVICE * ACCUM)) * EPOCHS
args = TrainingArguments(output_dir=str(OUTPUT_DIR / 'trainer'), num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
                         lr_scheduler_type='cosine', warmup_steps=max(1, round(WARMUP_SHARE * steps)),
                         per_device_train_batch_size=PER_DEVICE, gradient_accumulation_steps=ACCUM,
                         bf16=DTYPE == torch.bfloat16, fp16=DTYPE == torch.float16,
                         gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
                         logging_steps=10, save_strategy='steps', save_steps=50, save_total_limit=2,
                         report_to='none', seed=SEED, remove_unused_columns=False, optim='adamw_torch')
trainer = Trainer(model=model, args=args, train_dataset=Rows(train_encoded), data_collator=collate)
last = get_last_checkpoint(args.output_dir) if Path(args.output_dir).is_dir() else None
print('총', steps, 'step', '| 이어서 학습:', last)
result = trainer.train(resume_from_checkpoint=last)
(OUTPUT_DIR / 'train_log.json').write_text(json.dumps(trainer.state.log_history, indent=2))
(OUTPUT_DIR / 'train_metrics.json').write_text(json.dumps({**result.metrics, 'env': ENV}, indent=2))
ADAPTER_DIR = OUTPUT_DIR / 'adapter'
model.save_pretrained(ADAPTER_DIR, safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)
print('어댑터 저장', ADAPTER_DIR)

## 평가
validation 전체를 greedy로 생성해 형식·기능·인자 정확도를 잰다. 첫 32건은 한 건씩 생성한 결과와 같아야 묶음 생성을 쓴다.

In [ ]:
def generate(m, rows, batch_size):
    m.eval(); out = []
    dev = m.get_input_embeddings().weight.device
    order = sorted(range(len(rows)), key=lambda i: len(prompt_ids(rows[i])))
    texts = [None] * len(rows)
    with torch.no_grad():
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            prompts = [prompt_ids(rows[i]) for i in idx]
            n = max(map(len, prompts))
            ids = torch.tensor([[tokenizer.pad_token_id] * (n - len(p)) + p for p in prompts], device=dev)
            mask = torch.tensor([[0] * (n - len(p)) + [1] * len(p) for p in prompts], device=dev)
            gen = m.generate(input_ids=ids, attention_mask=mask, max_new_tokens=16, do_sample=False,
                             eos_token_id=[END_ID, BASE_EOS], pad_token_id=tokenizer.pad_token_id, use_cache=True)
            for i, seq in zip(idx, gen[:, n:].tolist()):
                cut = seq.index(END_ID) + 1 if END_ID in seq else next((j for j, t in enumerate(seq) if t in (BASE_EOS, tokenizer.pad_token_id)), len(seq))
                texts[i] = tokenizer.decode(seq[:cut], skip_special_tokens=False).strip()
    return texts

def score(rows, texts):
    cases = []
    for r, t in zip(rows, texts):
        got, gold = parse_call(t), parse_call(r['output'])
        cases.append(dict(case_id=r['case_id'], augmentation=r['augmentation'], user_text=r['user_text'],
                          expected=r['output'], actual=t, expected_function=r['function'],
                          actual_function=got[0] if got else None,
                          format_valid=got is not None, function_correct=bool(got) and got[0] == gold[0],
                          exact_correct=t == r['output']))
    def rate(cs, key):
        return round(sum(c[key] for c in cs) / max(1, len(cs)), 4)
    groups = lambda key: {v: dict(support=len(cs), function=rate(cs, 'function_correct'), exact=rate(cs, 'exact_correct'))
                          for v in sorted({c[key] for c in cases}) for cs in [[c for c in cases if c[key] == v]]}
    confusion = Counter(f"{c['expected_function']} -> {c['actual_function']}" for c in cases if not c['function_correct'])
    return cases, dict(support=len(cases), format_valid=rate(cases, 'format_valid'), function=rate(cases, 'function_correct'),
                       exact=rate(cases, 'exact_correct'), per_function=groups('expected_function'),
                       per_augmentation=groups('augmentation'), confusion=confusion.most_common(15))

model.config.use_cache = True
eval_rows = val_rows if EVAL_LIMIT is None else val_rows[:EVAL_LIMIT]
probe = eval_rows[:32]
BATCH = 16 if generate(model, probe, 1) == generate(model, probe, 16) else 1
print('묶음 생성 크기', BATCH)
val_texts = generate(model, eval_rows, BATCH)
val_cases, val_metrics = score(eval_rows, val_texts)
train_probe = random.Random(SEED).sample(train_rows, 300)
_, train_metrics = score(train_probe, generate(model, train_probe, BATCH))
metrics = dict(validation=val_metrics, train_sample_300=dict((k, train_metrics[k]) for k in ('format_valid', 'function', 'exact')),
               run=RUN_CONFIG | {'env': ENV, 'batch_generation': BATCH})
(OUTPUT_DIR / 'eval_metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
with (OUTPUT_DIR / 'eval_cases.jsonl').open('w', encoding='utf-8') as f:
    for c in val_cases:
        f.write(json.dumps(c, ensure_ascii=False) + '\n')
print(json.dumps({k: val_metrics[k] for k in ('support', 'format_valid', 'function', 'exact')}, ensure_ascii=False))
print('train 300', metrics['train_sample_300'])
for name, v in val_metrics['per_function'].items():
    print(f'{name:28s} n={v["support"]:4d} 기능 {v["function"]:.3f} 정확 {v["exact"]:.3f}')
print('자주 틀린 쌍', val_metrics['confusion'][:8])

## 실제 발화 평가
11/10 팀 테스트에서 받아 적힌 문장(문장 부호 제거)을 한 건씩 생성한다. 학습에 쓰지 않은 문장이다. 정답에 인자가 없는 행은 기능만 본다.

In [ ]:
def score_real(rows, texts):
    cases = []
    for r, t in zip(rows, texts):
        got = parse_call(t)
        cases.append(dict(case_id=r['case_id'], input_kind=r['augmentation'], user_text=r['user_text'], expected=r['output'],
                          expected_function=r['function'], actual=t, actual_function=got[0] if got else None,
                          format_valid=got is not None, function_correct=bool(got) and got[0] == r['function'],
                          exact_correct=None if r['output'] is None else t == r['output']))
    full = [c for c in cases if c['exact_correct'] is not None]
    rate = lambda cs, key: round(sum(c[key] for c in cs) / max(1, len(cs)), 4)
    by = lambda key: {v: dict(support=len(cs), function=rate(cs, 'function_correct'),
                              exact=rate([c for c in cs if c['exact_correct'] is not None], 'exact_correct'))
                      for v in sorted({c[key] for c in cases}) for cs in [[c for c in cases if c[key] == v]]}
    return cases, dict(support=len(cases), exact_support=len(full), format_valid=rate(cases, 'format_valid'),
                       function=rate(cases, 'function_correct'), exact=rate(full, 'exact_correct'),
                       per_function=by('expected_function'), per_input_kind=by('input_kind'))

real_cases, real_metrics = score_real(real_rows, generate(model, real_rows, 1))
(OUTPUT_DIR / 'real_speech_metrics.json').write_text(json.dumps(real_metrics | {'run': RUN_CONFIG}, ensure_ascii=False, indent=2))
with (OUTPUT_DIR / 'real_speech_cases.jsonl').open('w', encoding='utf-8') as f:
    for c in real_cases:
        f.write(json.dumps(c, ensure_ascii=False) + '\n')
print('실제 발화', {k: real_metrics[k] for k in ('support', 'exact_support', 'format_valid', 'function', 'exact')})
for name, v in real_metrics['per_function'].items():
    print(f'{name:28s} n={v["support"]:3d} 기능 {v["function"]:.3f} 정확 {v["exact"]:.3f}')

## 저장한 어댑터 다시 불러오기 확인

In [ ]:
import gc
del trainer, model, base; gc.collect(); torch.cuda.empty_cache()
reload_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
assert all(reload_tok.encode(t, add_special_tokens=False) == [i] for t, i in TOKEN_IDS.items())
reloaded = PeftModel.from_pretrained(load_base(), ADAPTER_DIR, is_trainable=False)
check = eval_rows[:64]
same = generate(reloaded, check, BATCH) == val_texts[:64]
(OUTPUT_DIR / 'reload_check.json').write_text(json.dumps({'cases': len(check), 'outputs_equal': same}))
print('다시 불러온 결과가 같은가:', same)
print('완료. 결과 폴더:', OUTPUT_DIR)